# 10. Building a small stock-research harness

**A harness is the code around a language model that lets it do useful work.** It supplies instructions, connects the model to Python functions and data, carries information between questions, and saves the result.

Why do we want one? Suppose we ask, **“How did AAPL perform from January through September 2024?”** A model can write an explanation, but it cannot see the database on our laptop just because we mention it. We need code to read the prices, calculate the return, and give that result to the model. If we then ask “What about MSFT?”, we also need to carry the earlier dates into the next question. Those connections are the harness.

In Lecture 2, we used pandas to turn data into an answer. In Lecture 4, we queried a database and ran stock reports for different tickers. We will reuse those ideas to build a small assistant that answers a stock-performance question, handles a follow-up, and writes a report on three stocks.

## The four parts of the harness

![The harness connects instructions and a model, tools and data, conversation memory, and a workflow that saves a report.](images/harness-overview.png)

| Part | What it does | Our version |
|---|---|---|
| Instructions and model | Understand the question and decide what to ask a tool to do | A short instruction and one Pydantic AI `Agent` |
| Tools and data | Carry out the calculation and return the evidence | One Python function reading the course database |
| Memory | Supply the earlier conversation when we ask a follow-up | A list of messages |
| Workflow and output | Decide which questions to run and keep the results | A `for` loop and a Markdown report |

The **model** proposes a tool call or writes an answer. The **tool** is a Python function that does the actual work. The **agent loop** connects them: ask the model, run a requested tool, give the result back, and continue until there is an answer. Pydantic AI supplies that loop. We build our harness by connecting it to our data, history, and reporting code.

We will add one piece at a time. First we will calculate an answer ourselves. Then we will let the agent request that calculation. Once one question works, we will add a follow-up and a small report. By the end, every box in the diagram will correspond to code we have run.

## 0. Environment and working directory

This notebook runs from the **repository root**, using the same `.venv` kernel as the other lectures. This keeps the database path consistent throughout the notebook.

The environment is installed with `uv sync --locked` from a terminal at the repository root. The notebook uses that repository's `.venv` as its kernel; an already running kernel needs a restart after installation. The environment includes Pydantic AI and both model providers used below. The interpreter path printed by the first cell identifies the active Python environment if an import fails.

The first cell finds the repository root even if the editor starts the kernel in this notebook's folder. It also locates `data.db`, the database distributed through bCourses for Lecture 4, at `Lectures/Lecture 4/data/data.db`.

In [ ]:
from pathlib import Path
import os
import sys

working_directory = Path.cwd().resolve()
ROOT = next(path for path in [working_directory, *working_directory.parents]
            if (path / 'Lectures' / 'Lecture 10').is_dir())
os.chdir(ROOT)
LECTURE_DIR = ROOT / 'Lectures' / 'Lecture 10'
DB_PATH = ROOT / 'Lectures' / 'Lecture 4' / 'data' / 'data.db'
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Download the course database from bCourses to {DB_PATH}')

print('Working directory:', Path.cwd())
print('Python interpreter:', sys.executable)
print('Database:', DB_PATH.relative_to(ROOT))

Pandas and SQLite handle the data work, while Pydantic AI manages the agent. `Markdown` displays the written answers inside the notebook. The helper `show_messages` turns the agent's messages into a readable table. The research logic appears in the cells below.

Model access requires an `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in a local `.env` file at the repository root. Only the key for the selected provider is needed. `load_dotenv(ROOT / '.env')` loads these settings into the Python session, where Pydantic AI can use them. The notebook contains the model name; the keys stay in the local file. Each agent run sends a request to the selected provider.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from dotenv import load_dotenv
from pydantic_ai import Agent

sys.path.insert(0, str(LECTURE_DIR))
from classroom_support import show_messages

load_dotenv(ROOT / '.env')
pd.set_option('display.max_colwidth', None)

## 1. Research question and price data

Before involving a model, we need a calculation we can check. Our first question has three inputs: a ticker, a start date, and an end date. We will answer it with the same database and pandas operations used in Lecture 4.

This gives us a result to compare with the assistant's answer later. The question concerns historical performance over a chosen period, so the first and last adjusted closing prices are enough for the calculation.

In [ ]:
TICKER = 'AAPL'
START_DATE = '2024-01-02'
END_DATE = '2024-09-30'

question = f'How did {TICKER} perform from {START_DATE} to {END_DATE}?'
print(question)

### Loading prices

The `ohlc` table stores price observations by ticker and date. The query selects `ts` and `adj_close`, filters to our stock and period, and sorts by date. The `?` placeholders receive the values in `params`; the SQL itself stays the same when the ticker changes.

The query belongs in `load_prices` because both a direct calculation and an agent's tool call will need it. The connection opens the course database for reading. The short checks at the end catch too little data, duplicate dates, and unusable prices before the return calculation. These are the same kinds of data checks introduced in Lecture 4.

In [ ]:
def load_prices(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    if pd.Timestamp(start_date) > pd.Timestamp(end_date):
        raise ValueError('The start date must come before the end date.')

    query = """
        SELECT ts, adj_close
        FROM ohlc
        WHERE ticker = ? AND ts >= ? AND ts <= ?
        ORDER BY ts
    """
    with sqlite3.connect(DB_PATH.as_uri() + '?mode=ro', uri=True) as connection:
        prices = pd.read_sql_query(
            query, connection,
            params=(ticker.upper(), start_date, end_date), parse_dates=['ts'],
        )

    if len(prices) < 2:
        raise ValueError(f'Need at least two prices for {ticker} in this period.')
    if prices['ts'].duplicated().any():
        raise ValueError('Check duplicate dates before calculating a return.')
    if not np.isfinite(prices['adj_close']).all() or (prices['adj_close'] <= 0).any():
        raise ValueError('Check missing or invalid adjusted prices.')
    return prices

prices = load_prices(TICKER, START_DATE, END_DATE)
display(prices.head())
display(prices.tail())

### The price series

The chart shows the adjusted prices selected by the query. The first and last observations determine the return. The answer will include those actual dates, since the requested range may start or end on a day with no observation.

In [ ]:
ax = prices.plot(x='ts', y='adj_close', figsize=(9, 3), legend=False,
                 title=f'{TICKER}: the prices behind our answer')
ax.set(xlabel='Date', ylabel='Adjusted close')
plt.show()

Changing `TICKER` from `AAPL` to `MSFT` selects a different price series through the same query and plotting code. The inputs change; the method stays the same. The examples below use AAPL for the initial question and MSFT for a later comparison.

With the prices available, the next part combines the query and return calculation into a reusable function.

## 2. A reusable stock-summary function

The assistant needs a useful action it can request. We will give it one: **summarize a stock's return over a date range**. Keeping the query and calculation together means the model can ask a sensible research question without constructing SQL or calculating the return itself.

`stock_summary` is an ordinary Python function. It calls `load_prices`, calculates `last / first - 1`, and returns a dictionary. The dictionary includes the ticker, actual dates, and observation count so the answer has context. As in Lecture 4, we keep the return as a fraction: `0.10` means 10%.

The `: str` annotations describe the three inputs as strings. `-> dict` describes the output. The text between triple quotes is the function's **docstring**: it explains the function and its arguments. Those descriptions will also help the model understand this tool.

In [ ]:
def stock_summary(ticker: str, start_date: str, end_date: str) -> dict:
    """Calculate a stock's historical return from the course database.

    Args:
        ticker: Stock symbol, such as AAPL or MSFT.
        start_date: First requested date, in YYYY-MM-DD format.
        end_date: Last requested date, in YYYY-MM-DD format.

    Returns:
        Actual dates, observation count, and total_return as a fraction.
    """
    prices = load_prices(ticker, start_date, end_date)
    first_price = prices['adj_close'].iloc[0]
    last_price = prices['adj_close'].iloc[-1]

    return {
        'ticker': ticker.upper(),
        'first_date': prices['ts'].iloc[0].strftime('%Y-%m-%d'),
        'last_date': prices['ts'].iloc[-1].strftime('%Y-%m-%d'),
        'observations': len(prices),
        'total_return': float(last_price / first_price - 1),
        'source': 'Course data.db / ohlc / adj_close',
    }

A direct call to `stock_summary` establishes the result the assistant should use. The percentage formatting below changes how the number is displayed; it leaves the stored fraction unchanged.

In [ ]:
summary = stock_summary(TICKER, START_DATE, END_DATE)
display(pd.Series(summary, name='Our Python result'))
print(f"{summary['ticker']} returned {summary['total_return']:.2%}.")

Python calculates the return with `last_price / first_price - 1`. The model's role is to recognize when this function can answer a question, choose its arguments, and explain the returned result. Registering the function as a tool connects those two responsibilities.

## 3. Connecting the function to an agent

So far, we have chosen and called the function ourselves. The next step is to let a model request it in response to our question. Pydantic AI will run the function and give the result back to the model.

![A question goes to the model, the model requests stock_summary, Python reads the database and calculates the result, and the model uses that result to answer.](images/tool-loop.png)

### Model selection

We will use a real model for every question. It receives our instructions and the description of `stock_summary`, then decides whether to answer directly, ask for missing information, or request the tool.

The default below is Claude Haiku 5.5. The OpenAI alternative is `model = 'openai:gpt-6-luna'`. Both providers are included in the root environment. Pydantic AI reads `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` from the environment according to the model's prefix.

`os.getenv` lets us optionally choose a model through `RESEARCH_MODEL` in the root `.env`; if that setting is absent, it uses the default shown below. Changing the model does not require changing our tool. A model change takes effect when the agent-creation cell runs again.

In [ ]:
model = os.getenv('RESEARCH_MODEL', 'anthropic:claude-haiku-5-5')
# To use OpenAI instead: model = 'openai:gpt-4.1-mini'
print('Model:', model)

### Creating the agent and registering the tool

`Agent(...)` creates the object that will manage the conversation with the model. We give it the selected model and a short instruction explaining the task. The instruction tells it to use our database calculation, ask for missing inputs, and write a short answer grounded in the result.

`research_agent.tool_plain(stock_summary)` **registers** the function: it adds `stock_summary` to the tools this agent can request. It does not call the function yet. Pydantic AI uses the function's name, annotations, and docstring to describe it to the model. “Plain” means it takes ordinary function arguments; we do not need an extra framework context argument.

You may also see `@research_agent.tool_plain` above a function definition. That decorator registers the function at definition time. Here we register the function we have already written and checked.

In [ ]:
research_agent = Agent(
    model,
    instructions=(
        'Help a student investigate historical stock returns. '
        'Use stock_summary when a question needs actual stock returns. '
        'Answer general conceptual questions directly, without a tool. '
        'For a calculation, use the ticker and dates in the question or history; '
        'if any are still missing, ask the user before calculating. '
        'Translate company names and calendar periods into tool arguments. '
        'When reporting a return, include the ticker, actual dates, percentage, and source. '
        'Keep answers short. Base claims about specific stock performance on tool results; '
        'do not invent reasons for price changes.'
    ),
)
research_agent.tool_plain(stock_summary)

### The first answer

`run(question)` starts the agent loop. The `await` expression waits for the run to finish; Jupyter supports it directly. `result.output` contains the final written answer, and `display(Markdown(...))` renders that text in the notebook.

The question uses the same ticker and dates as the direct function call in Section 2. Its answer should therefore describe the same result, although the model may round the percentage or vary its wording.

In [ ]:
first_result = await research_agent.run(question)
display(Markdown(first_result.output))

### The exchange behind the answer

The final answer alone does not show the tool call that supplied its evidence. `new_messages()` returns the messages from this run, including the request and the tool's result. The `show_messages` helper turns them into a readable table; it does not run any research.

The exchange follows **our question → the model's requested function and arguments → Python's result → the model's answer**. The model may also produce text before requesting the tool. The table makes the division of work visible: the model chooses the request and writes the answer, while Python reads the database and calculates the return.

In [ ]:
display(show_messages(first_result.new_messages()))

### When a question needs no tool

Registering a tool makes it available to the model; it does not require a call on every question. The next example asks why adjusted closing prices are useful. That explanation does not require a query for a particular stock, so the model can answer directly.

The message table shows whether a tool was requested. This distinguishes an answer based on a database result from a general explanation supplied by the model.

In [ ]:
concept_question = 'Why do we use adjusted closing prices when calculating a return?'
concept_result = await research_agent.run(concept_question)
display(Markdown(concept_result.output))
display(show_messages(concept_result.new_messages()))

A question such as “How did Apple perform in the first half of 2024?” requires price data. The model can translate “Apple” into `AAPL` and the period into date arguments for `stock_summary`. The Python function remains the same even when the question uses different wording.

A complete question supplies the information needed for a tool call. A follow-up may leave some of that information implicit, which is why conversation history is the next part of the harness.

## 4. Conversation history for follow-up questions

After asking about AAPL, a natural follow-up is **“Compare that with MSFT over the same dates.”** The new sentence omits the dates and the earlier stock's result. Those details need to travel with the question.

This is our first form of **memory**: the earlier conversation. Creating an `Agent` object does not automatically attach every previous run to the next one. Our code chooses which messages to pass in.

The first call below sends the follow-up on its own. The model has the instructions and tool description, but it has not received the earlier question or result. The instructions tell it to ask for missing context before calculating.

In [ ]:
follow_up = 'Compare that with MSFT over the same dates.'
without_history = await research_agent.run(follow_up)
display(Markdown(without_history.output))

### Including the earlier messages

`first_result.all_messages()` gives us the first conversation, including the tool call and returned numbers. We store it in `history`, then pass it with the named argument `message_history=history`. Now the model has the dates and AAPL's result when it reads our follow-up.

![The follow-up alone lacks dates. Passing the earlier question, tool result, and answer alongside the follow-up supplies the missing context.](images/conversation-memory.png)

After the new answer, we replace `history` with `follow_up_result.all_messages()`. That list already includes the history we supplied plus the new exchange. This is the list we would use for another follow-up.

In [ ]:
history = first_result.all_messages()
follow_up_result = await research_agent.run(follow_up, message_history=history)
display(Markdown(follow_up_result.output))

history = follow_up_result.all_messages()

The table below contains only the **new** exchange. The dates in its tool call can come from the earlier conversation, even though they do not appear in the follow-up question itself.

In [ ]:
display(show_messages(follow_up_result.new_messages()))
print('Messages available for our next follow-up:', len(history))

Memory here is a Python list passed with a request. It does not change the model's training. Restarting the kernel clears the variable. A saved report survives a restart, but the model only sees its contents if our code later reads them and includes them in a request.

For this short conversation, keeping the messages is enough. The useful design question is **“What information does the next question need?”** This follow-up needs the dates and prior result. Replacing MSFT with JPM would change the requested stock while preserving the same need for context.

An independent stock report can instead start a fresh conversation with a complete question. That lets us reuse the agent for several separate research tasks.

## 5. Researching several stocks

In Lecture 4, we generated reports for several stocks by changing the parameters in a loop. The same approach works here: send one complete question per ticker and collect the written answers.

The **workflow** is the Python code deciding what runs next. The three stocks are already known, so a `for` loop is enough. Each question includes its own dates and starts with no conversation history. This keeps each note focused on its stock.

`WATCHLIST` and the two dates define the project. Changing these values changes the research tasks without changing the agent or its tool. A longer watchlist produces more model calls.

In [ ]:
WATCHLIST = ['AAPL', 'MSFT', 'JPM']
PROJECT_START = '2024-01-02'
PROJECT_END = '2024-09-30'

`notes` maps each ticker to its written answer. `rows` holds the numeric summaries for a comparison table. Calling the Python function directly for that table keeps the numbers tied to the calculation established earlier. It repeats the small query instead of extracting numbers from the written paragraph.

The loop runs one ticker at a time and displays each note after its question completes.

In [ ]:
notes = {}
rows = []

for ticker in WATCHLIST:
    prompt = f'How did {ticker} perform from {PROJECT_START} to {PROJECT_END}?'
    result = await research_agent.run(prompt)
    notes[ticker] = result.output
    rows.append(stock_summary(ticker, PROJECT_START, PROJECT_END))
    display(Markdown(f'### {ticker}\n{result.output}'))

The comparison table uses the pandas operations from earlier classes: collect dictionaries into a DataFrame, select columns, and sort by a metric. The actual dates appear alongside the returns because the ranking depends on the period covered by each stock's data. A separate display copy formats the percentages while keeping the original returns numeric for the report.

In [ ]:
comparison = pd.DataFrame(rows).set_index('ticker')
comparison = comparison[['first_date', 'last_date', 'observations', 'total_return']]
comparison = comparison.sort_values('total_return', ascending=False)
display_table = comparison.copy()
display_table['total_return'] = display_table['total_return'].map('{:.2%}'.format)
display(display_table)

Replacing a ticker with `NVDA` changes the results and potentially the ranking. The tool, instructions, and agent remain the same; the workflow supplies a different task.

These results currently live in Python variables. Saving a report makes them available after the notebook closes.

## 6. Saving the report

A useful research run should leave an artifact. We will combine our comparison table and notes into a Markdown file. Markdown is plain text with headings and tables; the same format is used in the repository's README files.

Python will build the report and write the file. The model has already supplied the short notes. We will include the requested period, the actual dates for each stock, and the database source so someone reading the file can understand what was analyzed.

The cell below formats one table row per stock and one section per note. `write_text` saves the assembled string. Rerunning it replaces this example report with the current results.

In [ ]:
report_parts = [
    '# Stock performance report',
    '',
    f'Requested period: {PROJECT_START} to {PROJECT_END}.',
    '',
    'Source: course data.db, ohlc table, adjusted closing prices.',
    '',
    '## Comparison',
    '',
    '| Ticker | First date | Last date | Prices | Return |',
    '|---|---|---|---:|---:|',
]
for ticker, row in comparison.iterrows():
    report_parts.append(
        f"| {ticker} | {row['first_date']} | {row['last_date']} | "
        f"{row['observations']} | {row['total_return']:.2%} |"
    )

report_text = '\n'.join(report_parts)
for ticker, note in notes.items():
    report_text += f'\n\n## {ticker}\n\n{note}'

output_directory = LECTURE_DIR / 'runs' / 'intro'
output_directory.mkdir(parents=True, exist_ok=True)
report_path = output_directory / 'stock_report.md'
report_path.write_text(report_text + '\n', encoding='utf-8')
print('Saved:', report_path.relative_to(ROOT))

The next cell reads the saved file and displays its contents. This is the report on disk, rather than a new model response. It contains the numeric comparison and the individual notes in a format that can also be opened in an editor.

In [ ]:
display(Markdown(report_path.read_text(encoding='utf-8')))

## 7. Exploring other questions

The harness now has all four parts: instructions and a model, a database tool, conversation memory, and a workflow that saves a report. The `my_question` variable below provides a place for a question in ordinary language.

The initial example says “Nvidia” and “the second quarter” instead of specifying `NVDA` and two dates. The message table exposes the model's chosen arguments and the returned data behind its answer.

Different questions call for different behavior:

| Question | Information needed |
|---|---|
| “Compare Apple and Microsoft from January 2 to September 30, 2024.” | Returns for both stocks, available through separate tool calls |
| “What does a 10% return mean?” | A general explanation, with no database query needed |
| “How did JPM perform?” | A date range, supplied through history or a clarifying question |

AAPL, MSFT, JPM, and NVDA have data for the first nine months of 2024 in the course database. The tool calculates historical returns; it does not retrieve news or identify the business events behind a price move.

In [ ]:
my_question = 'How did Nvidia perform during the second quarter of 2024?'
my_result = await research_agent.run(my_question)
display(Markdown(my_result.output))
display(show_messages(my_result.new_messages()))

A follow-up to this answer can receive `my_result.all_messages()` as its history. Without that history, the new question needs to supply its own ticker and dates. The same stocks and dates can also become the inputs to the reporting loop in Section 5.

The agent's instructions control the requested writing style as well as its use of tools. For example, requesting two bullet points changes the desired presentation of the answer while leaving the calculation in Python. Because the instructions are supplied when `research_agent` is created, a change takes effect when that cell runs again.

For a data question, the complete path is **question → model's tool request → Python calculation → tool result → written answer**. Conversation history supplies context for related questions, and the reporting code collects and saves the results.

A fixed report can still be written entirely in Python, as in Lecture 4. Adding a model provides a conversational interface and a written explanation. The harness connects that interface to the work our Python already knows how to do.

### References

The message-display helper is defined in [classroom_support.py](classroom_support.py). Pydantic AI's documentation covers [function tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/), [message history](https://pydantic.dev/docs/ai/core-concepts/message-history/), and connections to [Anthropic](https://pydantic.dev/docs/ai/models/anthropic/) and [OpenAI](https://pydantic.dev/docs/ai/models/openai/) models.